# 🖥️ Servidor remoto Z-Image + LLM (contrato img-v1 / OpenAI-compatible) — Colab gratis

Este notebook levanta un servidor que la app de escritorio puede usar cuando tu PC no tiene GPU compatible (ver **Ajustes → Servidores** en la app). Por defecto sirve **solo imagen** (lo más estable en Colab gratis); el LLM es opcional y se activa aparte (formulario de configuración — correr los dos a la vez es la causa más común de caídas por falta de RAM, ver aviso de recursos más abajo):

- **Imagen** — **Z-Image-Turbo** (Apache-2.0, público en Hugging Face), el mismo modelo que usa la app en local (único backend de imagen desde 08-Ago-2026).
- **LLM de prompts/guion** — `Qwen3-4B` (GGUF, también público), el mismo modelo que usa la app en local, servido por OpenAI-compatible (`/v1/chat/completions`). Corre en **CPU** (Z-Image ya usa toda la VRAM disponible de la T4, sin margen para el LLM también en GPU — ver la nota de la sección 4).

## 🟢 Cómo usarlo (sin saber programar)
1. Arriba: *Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU*** (gratis) y guarda.
2. En el **formulario de configuración** (más abajo) deja marcado lo que quieras: imágenes, textos, o ambos. Si quieres no re-descargar los pesos cada sesión, marca también **Guardar modelos en Google Drive** (te pedirá autorizar el acceso la primera vez).
3. Arriba: *Entorno de ejecución → **Ejecutar todas*** (o `Ctrl+F9`). Espera unos minutos la primera vez (descarga los modelos); si usas Drive, las siguientes veces es mucho más rápido.
4. Copia la **clave de API** (la muestra el formulario) y la **URL** `https://…trycloudflare.com` (la muestra la última celda, "▶️ Encender el servidor") en la app: *Ajustes → Servidores*.
5. Deja esta pestaña abierta mientras uses la app. Para parar: *Entorno de ejecución → Interrumpir ejecución*.

Contrato: ver `docs/EPIC_BACKEND_REMOTO.md` y `docs/SERVIDORES_REMOTOS.md` en el repo. Este notebook **no contiene nada propio de la app**: ni prompts, ni reglas de personaje, ni orquestación — solo carga los modelos públicos y los sirve.

> ⚠️ **Léelo antes de usarlo — bajo tu propio riesgo:**
> - **RAM del sistema (no VRAM) es el recurso más justo en Colab gratis** (~12-13 GB): si la sesión se cae sola con "resources spiked"/se reinicia sin aviso, casi seguro fue por RAM. Si te pasa, en el formulario desmarca uno de los dos modelos para cargar solo uno a la vez.
> - **Si el kernel se reinicia solo** (verás "kernel restarted" en los logs, o la celda del servidor deja de responder de golpe): fue un crash duro, no un error normal — vuelve a ejecutar desde la celda de configuración. Este notebook ya evita la causa conocida (`enable_model_cpu_offload()` + text_encoder en 4-bit) cargando Z-Image con VRAM completa sin offload y limitando la resolución máxima — si te sigue pasando con la resolución ya reducida, es un caso nuevo, repórtalo con el tamaño/pasos exactos.
> - El túnel (`cloudflared`) publica tu sesión de Colab en una URL **pública** mientras esta pestaña siga abierta. Cualquiera con esa URL puede generar imágenes/texto con tu cuota — no la compartas fuera de tu app. Deja marcada la **clave automática** del formulario para cerrarla con un secreto.
> - Colab gratis es **frágil**: se corta por inactividad (~90 min) o a las pocas horas de uso, y su ToS no está pensado para servir un backend persistente hacia una app externa — trata esto como **experimental**, no como un servidor estable.
> - Todo lo que la app envíe aquí (prompts de imagen, fragmentos de tu guion) sale de tu equipo hacia esta sesión de Colab. No subas nada sensible.
> - Guardar modelos en Drive los deja en tu Drive personal — puedes borrar la carpeta `again_modelos_cache` cuando quieras liberar ese espacio.

## 1. Instalar dependencias
Solo hace falta una vez por sesión. `diffusers`/`transformers`/`accelerate`/`gguf`/`bitsandbytes` para Z-Image; `llama-cpp-python` para el LLM; `fastapi`/`uvicorn` para servir ambos contratos.

In [ ]:
#@title  ⬇️  Instalar (ejecuta y espera, ~1-2 min) { display-mode: "form" }
!pip install -q -U diffusers transformers accelerate gguf bitsandbytes

!pip install -q fastapi "uvicorn[standard]" pydantic



# cloudflared: túnel "quick tunnel" (sin cuenta, sin token) para publicar el

# servidor local de Colab en una URL https:// real que la app pueda alcanzar.

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared

!chmod +x cloudflared

print('Listo.')

### Contrato img-v1 + LLM (autocontenido)
El código del servidor (`build_app`, ver `docs/EPIC_BACKEND_REMOTO.md`) se escribe AQUÍ mismo con `%%writefile` — este notebook no depende de clonar el repo ni de ninguna URL externa: es la misma implementación que usan los tests del proyecto (`tests/test_remote_server_contract.py`), pegada tal cual. `generate_fn`/`chat_fn` son ambos opcionales — si sólo cargas un modelo (celda 4), el otro simplemente no se monta.

In [ ]:
%%writefile server_img_v1.py
"""
remote/server_img_v1.py — Implementación reutilizable del contrato de imagen
`img-v1` (ver docs/EPIC_BACKEND_REMOTO.md) + LLM OpenAI-compatible, sobre FastAPI.

INOCUO A PROPÓSITO (principio de protección de IP del epic): este módulo no
sabe nada de la app — ni de personajes, ni de perfiles de prompt, ni de
orquestación. Sólo expone `GET /health` + `POST /generate` (+ `POST /refine`
si se pasa `refine_fn`, hires-fix opcional) — los que consume
`pipeline/backends/remote_backend.py::RemoteBackend` — y, si se pasa `chat_fn`,
`POST /v1/chat/completions` (el que consume
`pipeline/backends/remote_llm.py::RemoteLLM`). Toda la ingeniería de prompts
se queda del lado de la app de escritorio; aquí sólo entra un prompt/mensajes
ya construidos y sale una imagen/respuesta.

Lo usa `remote/colab_zimage_server.ipynb` (R3) para servir Z-Image-Turbo +
Qwen3-4B (el mismo LLM de prompts que usa la app en local) desde UN solo
notebook gratuito y UN solo túnel — el usuario pega la misma URL en "Imagen
remota" y "LLM remoto" (Ajustes → Servidores). `build_app` no depende de
Z-Image/Qwen en absoluto: cualquier servidor propio (otros modelos, otro
hardware) puede reusarlo pasando sus propios `generate_fn`/`chat_fn`.
"""
from __future__ import annotations

import base64
import io
import random
from typing import Any, Callable, Optional

from pydantic import BaseModel

CONTRATO = 'img-v1'

# Igual que RemoteBackend (ver remote_backend.py:_ESPERA_REINTENTO/_TIMEOUT_*),
# un seed fuera de este rango no tiene sentido para torch.Generator.manual_seed.
_SEED_MAX = 2 ** 31 - 1

GenerateFn = Callable[[str, str, int, int, int, float, int, Optional[int]], 'PIL.Image.Image']
# `messages` en formato OpenAI (`[{'role':..., 'content':...}, ...]`) → dict de
# respuesta YA en forma OpenAI (`{'choices': [{'message': {'content': ...}}], ...}`)
# — es exactamente lo que devuelve `llama_cpp.Llama.create_chat_completion(...)`
# de por sí, así que el `chat_fn` típico es ese método sin envolver nada más.
# `repeat_penalty` va incluido porque `pipeline/guion_ia.py::_llm` siempre lo
# manda (el modelo local llama_cpp.Llama lo soporta de forma nativa) — sin
# este parámetro aquí, el servidor no puede reenviarlo y la llamada del
# cliente (RemoteLLM) recibiría un cuerpo incompleto.
ChatFn = Callable[[list, float, float, int, float], dict]
# 2ª pasada (hires-fix) opcional para backends que la soporten (p.ej. SD1.5) —
# `img` YA viene decodificada (el módulo se encarga de base64<->PIL, igual que
# con `/generate`); `strength` la calcula el CLIENTE (RemoteBackend, misma
# fórmula que el hires-fix local) y viaja como número — este módulo nunca ve
# esa fórmula, sólo el resultado.
RefineFn = Callable[['PIL.Image.Image', str, str, float, int, float, int], 'PIL.Image.Image']


# Definida a NIVEL DE MÓDULO (no dentro de `build_app`) a propósito: con
# `from __future__ import annotations` las anotaciones quedan como strings, y
# FastAPI las resuelve vía `typing.get_type_hints()` contra los globals DEL
# MÓDULO donde vive la función de ruta — una clase local a `build_app` no
# sería visible ahí, y el endpoint `/generate` trataría `body: GenerateBody`
# como parámetro de query en vez de request body (falla en silencio con 422
# "Field required", no un error obvio de import).
class GenerateBody(BaseModel):
    prompt: str
    negative: str = ''
    width: int
    height: int
    steps: int
    guidance: float
    seed: int | None = None
    max_seq_len: int | None = None


class ChatBody(BaseModel):
    model: str | None = None   # ignorado: este servidor sirve un único modelo
    messages: list[dict[str, Any]]
    temperature: float = 0.3
    top_p: float = 0.85
    max_tokens: int = 128
    repeat_penalty: float = 1.1


class RefineBody(BaseModel):
    image_b64: str
    prompt: str
    negative: str = ''
    strength: float
    steps: int
    guidance: float
    seed: int = 0


def build_app(model_name: str, generate_fn: GenerateFn | None = None,
              chat_fn: ChatFn | None = None, refine_fn: RefineFn | None = None,
              api_key: str | None = None):
    """Construye la app FastAPI que sirve `img-v1` y, opcionalmente, un LLM
    OpenAI-compatible — en el MISMO puerto/URL, para que el usuario sólo
    necesite un túnel y pueda pegar la misma URL en "Imagen remota" y "LLM
    remoto" (Ajustes → Servidores).

    `generate_fn(prompt, negative, width, height, steps, guidance, seed,
    max_seq_len) -> PIL.Image.Image` — la pieza que varía entre servidores de
    imagen; el notebook Z-Image le pasa un wrapper sobre `ZImagePipeline`.
    `None` = no se monta `/generate` (servidor sólo-LLM).

    `chat_fn(messages, temperature, top_p, max_tokens, repeat_penalty) -> dict` — respuesta ya
    en forma OpenAI; el notebook le pasa `llama_cpp.Llama.create_chat_completion`
    directo, sin envolver nada (esa firma ya devuelve exactamente esa forma).
    `None` = no se monta `/v1/chat/completions` (servidor sólo-imagen, R1/R3
    originales).

    `refine_fn(img, prompt, negative, strength, steps, guidance, seed) ->
    PIL.Image.Image` — 2ª pasada (hires-fix) OPCIONAL, sólo para backends que
    la soporten (p.ej. SD1.5). `None` (default) = no se monta `/refine` —
    `RemoteBackend.refinar()` cae de vuelta a la imagen base sin refinar, sin
    romper nada (mismo criterio que un backend sin hires-fix en local). Es
    ADITIVO al contrato `img-v1`: no cambia `/generate` ni `/health`, así que
    un cliente viejo (sin saber de `/refine`) sigue funcionando igual.

    `api_key`: si se da, TODOS los endpoints montados exigen
    `Authorization: Bearer <api_key>` (mismo header que mandan
    `RemoteBackend`/`RemoteLLM`). `None`/`''` = servidor abierto — sólo
    razonable detrás de un túnel que el propio usuario controla.
    """
    from fastapi import FastAPI, Header, HTTPException
    from PIL import Image

    app = FastAPI(title='img-v1 + LLM server', version=CONTRATO)

    def _exigir_auth(authorization: str | None) -> None:
        if api_key and authorization != f'Bearer {api_key}':
            raise HTTPException(status_code=401, detail='API key inválida o ausente.')

    @app.get('/health')
    def health(authorization: str | None = Header(default=None)):
        _exigir_auth(authorization)
        return {'ok': True, 'model': model_name, 'contract': CONTRATO}

    if generate_fn is not None:
        @app.post('/generate')
        def generate(body: GenerateBody, authorization: str | None = Header(default=None)):
            _exigir_auth(authorization)
            seed = body.seed if body.seed is not None else random.randint(0, _SEED_MAX)
            img = generate_fn(body.prompt, body.negative, body.width, body.height,
                              body.steps, body.guidance, seed, body.max_seq_len)
            buf = io.BytesIO()
            img.convert('RGB').save(buf, format='PNG')
            image_b64 = base64.b64encode(buf.getvalue()).decode('ascii')
            return {'image_b64': image_b64, 'seed': seed, 'contract': CONTRATO}

    if refine_fn is not None:
        @app.post('/refine')
        def refine(body: RefineBody, authorization: str | None = Header(default=None)):
            _exigir_auth(authorization)
            img = Image.open(io.BytesIO(base64.b64decode(body.image_b64))).convert('RGB')
            out = refine_fn(img, body.prompt, body.negative, body.strength,
                            body.steps, body.guidance, body.seed)
            buf = io.BytesIO()
            out.convert('RGB').save(buf, format='PNG')
            image_b64 = base64.b64encode(buf.getvalue()).decode('ascii')
            return {'image_b64': image_b64, 'contract': CONTRATO}

    if chat_fn is not None:
        @app.post('/v1/chat/completions')
        def chat_completions(body: ChatBody, authorization: str | None = Header(default=None)):
            _exigir_auth(authorization)
            return chat_fn(body.messages, body.temperature, body.top_p, body.max_tokens,
                           body.repeat_penalty)

        @app.get('/v1/models')
        def models(authorization: str | None = Header(default=None)):
            _exigir_auth(authorization)
            return {'data': [{'id': model_name, 'object': 'model'}]}

    return app


def lanzar_con_tunel(app, puerto: int = 8000, log=print) -> None:
    """Arranca `app` en un hilo (uvicorn) y abre un túnel cloudflared "quick
    tunnel" (sin cuenta, sin token) apuntando a `http://localhost:{puerto}`,
    imprimiendo la URL pública en cuanto cloudflared la anuncia. Pensado para
    la ÚLTIMA celda del notebook — bloquea (join) hasta que se interrumpa la
    sesión, igual que cualquier servidor de desarrollo.

    Requiere el binario `cloudflared` ya descargado en el cwd (lo deja listo
    la celda de instalación del notebook) — este módulo no lo descarga para
    no acoplar la librería a una plataforma concreta (Colab vs servidor
    propio, donde el usuario puede preferir su propio túnel/dominio)."""
    import re
    import subprocess
    import threading

    import uvicorn

    config = uvicorn.Config(app, host='0.0.0.0', port=puerto, log_level='warning')
    server = uvicorn.Server(config)
    hilo_servidor = threading.Thread(target=server.run, daemon=True)
    hilo_servidor.start()

    proceso = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', f'http://localhost:{puerto}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log('Abriendo túnel (cloudflared)… puede tardar unos segundos.')
    patron_url = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
    try:
        for linea in proceso.stdout:
            m = patron_url.search(linea)
            if m:
                log(f'\nServidor listo. Pega esta URL en Ajustes → Servidores → Imagen remota:\n'
                    f'  {m.group(0)}\n')
                break
        # sigue vivo hasta que se interrumpa la celda (Runtime → Interrupt)
        proceso.wait()
    except KeyboardInterrupt:
        proceso.terminate()


## 2. Elige qué activar (con casillas)
La celda de abajo es un **formulario**: por defecto solo **Generar imágenes** viene marcado — es el modo recomendado, el más estable en la GPU/RAM gratuita de Colab. Marca también **Generar textos de guion** si tu sesión aguanta los dos a la vez (o si no vas a generar imágenes ahora); si quieres proteger el servidor con una **clave automática**; y si quieres **guardar los modelos en tu Google Drive** para no volver a descargarlos cada sesión. No hay que escribir código.

Si activaste los dos y la sesión se cae por recursos, vuelve a desmarcar **Generar textos de guion** y usa solo imagen. Si dejaste ambos marcados, la misma URL sirve para los dos.

In [ ]:
#@title ⚙️ Configuración — marca qué activar y ejecuta todo { display-mode: "form" }
#@markdown Marca lo que quieras usar y luego pulsa arriba **Entorno de ejecución → Ejecutar todas** (o `Ctrl+F9`). No necesitas tocar nada más.
#@markdown &nbsp;

Generar_imagenes = True  #@param {type:"boolean"}
#@markdown Recomendado: deja esto DESACTIVADO — correr el LLM junto con Z-Image es la causa más común de caídas por falta de RAM en la sesión gratuita de Colab. Actívalo solo si tu sesión lo aguanta (o si no vas a cargar imágenes).
Generar_textos_de_guion = False  #@param {type:"boolean"}

#@markdown ---
#@markdown Deja esto marcado para proteger tu servidor con una clave secreta automática (recomendado). La clave se muestra al ejecutar esta celda: cópiala en la app.
Proteger_con_clave_automatica = True  #@param {type:"boolean"}

#@markdown ---
#@markdown Guarda los pesos descargados en tu Google Drive para no volver a descargarlos en la próxima sesión — te pedirá autorizar el acceso a Drive la primera vez. Ahorra varios minutos en cada reinicio, a costa de ese espacio en tu Drive.
Guardar_modelos_en_Google_Drive = False  #@param {type:"boolean"}

import os
import secrets

# Reduce fallos por FRAGMENTACIÓN de VRAM (no por falta real de memoria) en
# sesiones largas con muchas generaciones seguidas — hay que fijarlo ANTES de
# que 'torch' se importe en cualquier celda de más abajo (lee esta variable
# una sola vez, al arrancar CUDA).
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CARGAR_IMAGEN = Generar_imagenes           # Z-Image-Turbo (único backend de imagen)
CARGAR_LLM = Generar_textos_de_guion       # Qwen3-4B, prompts/guion (CPU)
API_KEY = ('sk-' + secrets.token_hex(16)) if Proteger_con_clave_automatica else ''

if Guardar_modelos_en_Google_Drive:
    from google.colab import drive
    drive.mount('/content/drive')
    _CACHE_DIR = '/content/drive/MyDrive/again_modelos_cache'
    os.makedirs(_CACHE_DIR, exist_ok=True)
    # Hay que fijar esto ANTES de que las celdas de más abajo importen
    # huggingface_hub (celdas 8 y 10) — leen estas variables al importarse.
    os.environ['HF_HOME'] = _CACHE_DIR
    os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(_CACHE_DIR, 'hub')
    print(f'📁 Modelos se guardarán/leerán desde tu Drive: {_CACHE_DIR}')
    print('   (la próxima vez que ejecutes este notebook, la descarga se salta si ya están ahí)')

print('Imágenes (Z-Image-Turbo):', '✅ activadas' if CARGAR_IMAGEN else '⬜ desactivadas')
print('Textos de guion:', '✅ activados' if CARGAR_LLM else '⬜ desactivados')
if API_KEY:
    print('\n🔑 Clave de API (cópiala en la app, campo «Clave de API» de cada servicio):\n   ' + API_KEY)
else:
    print('\n⚠️ Servidor SIN clave: cualquiera con la URL podrá usar tu sesión. Márcala si vas a dejarlo corriendo.')

## 3. Cargar el generador de imágenes (GPU)
Se salta entero si desmarcaste **Generar imágenes** arriba.

**Z-Image-Turbo** — transformer Q4 GGUF (comunidad, `unsloth/Z-Image-Turbo-GGUF`) + text encoder Qwen3-4B en 4-bit. **VRAM completa, SIN offload** — probado en real que `enable_model_cpu_offload()` combinado con el text_encoder en 4-bit (bitsandbytes) puede **crashear el proceso entero** (reinicio del kernel de Colab, sin excepción capturable) al mover tensores cuantizados entre CPU y GPU. La resolución de producción (1536×864) excede la VRAM de una T4 con este approach, así que la celda "▶️ Encender el servidor" limita la resolución máxima que se le pide al modelo (genera más pequeño y deja que la app reescale).

En la app local Z-Image usa sus propios ajustes ya calibrados (offload sí se usa correctamente ahí, con más VRAM disponible) — no copies estos cambios al backend local, son específicos de las limitaciones de la T4 gratuita de Colab.

In [ ]:
#@title  ⬇️  Cargar el generador de imágenes (no toques nada, solo ejécutala) { display-mode: "form" }
pipe = None

if CARGAR_IMAGEN:
    import torch
    from diffusers import (GGUFQuantizationConfig, ZImagePipeline, ZImageTransformer2DModel)
    from transformers import BitsAndBytesConfig, Qwen3Model
    from huggingface_hub import hf_hub_download

    ZIMAGE_REPO = 'Tongyi-MAI/Z-Image-Turbo'
    ZIMAGE_GGUF_REPO = 'unsloth/Z-Image-Turbo-GGUF'
    ZIMAGE_GGUF_FILE = 'z-image-turbo-Q4_K_M.gguf'
    MAX_SEQ_LEN_DEFAULT = 1024
    DEVICE = 'cuda'
    DTYPE = torch.bfloat16

    gguf_path = hf_hub_download(ZIMAGE_GGUF_REPO, ZIMAGE_GGUF_FILE)
    transformer = ZImageTransformer2DModel.from_single_file(
        gguf_path, config=ZIMAGE_REPO, subfolder='transformer',
        quantization_config=GGUFQuantizationConfig(compute_dtype=DTYPE),
        torch_dtype=DTYPE,
    )
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
                                    bnb_4bit_quant_type='nf4')
    text_encoder = Qwen3Model.from_pretrained(
        ZIMAGE_REPO, subfolder='text_encoder', quantization_config=bnb_config, torch_dtype=DTYPE,
    )
    pipe = ZImagePipeline.from_pretrained(
        ZIMAGE_REPO, transformer=transformer, text_encoder=text_encoder, torch_dtype=DTYPE,
    )
    # NO offload: probado en real que enable_model_cpu_offload() combinado con el
    # text_encoder en 4-bit (bitsandbytes) puede CRASHEAR el proceso entero (kernel
    # restart, sin excepción capturable) al mover tensores cuantizados entre CPU y
    # GPU — incompatibilidad conocida accelerate+bitsandbytes, peor que un simple
    # OOM: con offload el servidor entero muere; con VRAM completa, un fallo por
    # falta de memoria es una excepción normal de Python y el servidor sigue vivo
    # para la siguiente petición. Por eso aquí se prefiere VRAM completa + limitar
    # la resolución máxima en generate_fn (ver celda de "Encender el servidor") en
    # vez de offload.
    pipe.to(DEVICE)
    # Calentamiento (15-Ago-2026): la PRIMERA llamada de inferencia tras cargar
    # el pipeline es bastante más lenta que las siguientes (autotune/compilación
    # de kernels CUDA la primera vez) — probado en real que ese costo extra puede
    # hacer que la primera petición de un usuario real tarde mucho más de lo
    # normal (llegó a coincidir con un 524 del túnel). Se absorbe aquí, ANTES de
    # abrir el túnel (celda siguiente), con una generación mínima descartada —
    # así el usuario nunca paga ese costo de arranque.
    import time as _time
    _t0 = _time.time()
    pipe(prompt='a red apple', height=256, width=256, num_inference_steps=1,
         guidance_scale=0.0, generator=torch.Generator(DEVICE).manual_seed(0),
         max_sequence_length=MAX_SEQ_LEN_DEFAULT)
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Z-Image-Turbo cargado y calentado (VRAM completa, {_time.time()-_t0:.0f}s de warmup).')
else:
    print('CARGAR_IMAGEN=False — se salta esta celda, no se monta /generate.')

## 4. Cargar el LLM de prompts (Qwen3-4B)
Se salta entero si desmarcaste **Generar textos de guion** arriba. Mismo modelo que usa la app en local (`pipeline/models.py::_cargar_qwen`).

> **CPU si Z-Image está activo, GPU si no.** Z-Image ya usa toda la VRAM disponible de la T4 (~14.5 GB) sin margen para el LLM también en GPU — ponerlo ahí causó `CUDA OutOfMemoryError` en pruebas reales. Si sólo activaste el LLM (sin imágenes), va a GPU — no hay nada compitiendo por VRAM, mucho más rápido que CPU (~100s/bloque en CPU → segundos en GPU).

In [ ]:
#@title  ⬇️  Cargar el generador de textos (no toques nada, solo ejécutala) { display-mode: "form" }
llm = None
if CARGAR_LLM:
    import os
    from huggingface_hub import hf_hub_download

    PROMPT_LLM_GGUF_REPO = 'Qwen/Qwen3-4B-GGUF'
    PROMPT_LLM_GGUF_FILE = 'Qwen3-4B-Q4_K_M.gguf'
    ruta_llm = hf_hub_download(PROMPT_LLM_GGUF_REPO, PROMPT_LLM_GGUF_FILE)

    # LLM en GPU sólo si NO hay Z-Image cargado (usa toda la VRAM disponible de la
    # T4 sin margen) — ponerlo en GPU junto a Z-Image causó CUDA OutOfMemoryError
    # en pruebas reales.
    _LLM_EN_GPU = not CARGAR_IMAGEN

    if _LLM_EN_GPU:
        # Wheel CUDA precompilada (evita compilar desde fuente, 10+ min en Colab).
        !pip install -q llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
        from llama_cpp import Llama
        llm = Llama(model_path=ruta_llm, n_ctx=4096, n_gpu_layers=-1, verbose=False)
        print('LLM de prompts cargado en GPU (sin imagen activa, VRAM libre).')
    else:
        # Wheel precompilada de PyPI (CPU) — Z-Image ya usa toda la VRAM disponible.
        !pip install -q llama-cpp-python
        from llama_cpp import Llama
        llm = Llama(model_path=ruta_llm, n_ctx=4096, n_gpu_layers=0, verbose=False)
        print('LLM de prompts cargado en CPU (Z-Image activo, sin margen de VRAM).')
else:
    print('CARGAR_LLM=False — se salta esta celda, no se monta /v1/chat/completions.')

## 5. Servir y abrir el túnel
Define `generate_fn`/`chat_fn` SÓLO para lo que se haya cargado arriba (`pipe`/`llm` distintos de `None`) y los monta en un único servidor. Al cabo de unos segundos aparece una URL `https://xxxxx.trycloudflare.com` — esa URL va en **Ajustes → Servidores** de la app: en **Imagen remota** si activaste imágenes, en **LLM remoto** si activaste textos (misma URL para ambos si activaste los dos).

Esta celda **se queda corriendo** (es el servidor). Para pararlo: *Entorno de ejecución → Interrumpir ejecución*.

In [ ]:
#@title  ▶️  Encender el servidor (aquí sale tu URL — déjala corriendo) { display-mode: "form" }
from server_img_v1 import build_app, lanzar_con_tunel

assert pipe is not None or llm is not None, (
    'CARGAR_IMAGEN y CARGAR_LLM son False los dos — no hay nada que servir. '
    'Activa al menos uno en el formulario de configuración y vuelve a ejecutar.')

# Tope de píxeles: por VELOCIDAD, no solo VRAM. Medido en real 15-Ago-2026: a
# 1024x576/6 pasos, ~120-128s/img — muy lento para uso interactivo. El tiempo
# escala ~linealmente con píxeles × pasos (confirmado con 2 puntos reales: 46s a
# 768x432/4 pasos, 124s a 1024x576/6 pasos — mismo ratio ×~2.67 en ambos ejes).
# La app (`desktop/engine_worker.py::_config_imagen`) pide 4 pasos en vez de 6
# cuando el backend remoto está activo (15-Ago) — con eso, este tope se fijó a
# 576x324 para caer en ~26s/img estimados (extrapolado del modelo lineal de
# arriba; vuelve a medir con el servidor real tras desplegar este cambio antes
# de darlo por bueno). Si la app pide más resolución, se genera más pequeño
# (mismo aspect ratio) y se deja que la app reescale al tamaño final (ya lo hace
# siempre — RemoteBackend.finalizar en pipeline/backends/remote_backend.py) —
# ojo, a costa de nitidez: 576x324 es ~14% de los píxeles nativos (1536x864).
# Si necesitas más calidad y toleras más tiempo por imagen, sube este número —
# vuelve a medir el tiempo real antes de subirlo mucho.
#
# Nota histórica (ya no aplica): se llegó a pensar que había un límite DURO de
# ~100s en el túnel gratuito (el primer 524 de la sesión de pruebas coincidió con
# ese tiempo) — pero al repetir la prueba con más trabajo (1024x576/6 pasos) el
# request terminó bien dos veces seguidas en 120s y 128s. El 524 fue casi seguro
# un COLD START de la primera inferencia tras arrancar el servidor (autotune de
# kernels CUDA) — por eso se agregó un calentamiento en la celda de carga
# (sección 3), no por este tope. Si SIGUES viendo 524 en uso normal (no en la
# primera petición tras arrancar), repórtalo con la duración medida.
_MAX_PIXELS_ZIMAGE = 576 * 324


def _clamp(w, h, tope_px):
    if w * h <= tope_px:
        return w, h
    escala = (tope_px / (w * h)) ** 0.5
    return max(16, round(w * escala / 16) * 16), max(16, round(h * escala / 16) * 16)


generate_fn = None
if pipe is not None:
    def generate_fn(prompt, negative, width, height, steps, guidance, seed, max_seq_len):
        w, h = _clamp(width, height, _MAX_PIXELS_ZIMAGE)
        generator = torch.Generator(DEVICE).manual_seed(seed)
        kw = dict(prompt=prompt, height=h, width=w, num_inference_steps=steps,
                  guidance_scale=guidance, generator=generator,
                  max_sequence_length=max_seq_len or MAX_SEQ_LEN_DEFAULT)
        if guidance > 1.0:  # perfil turbo destilado (cfg~0): ignora negative si no se sube guidance
            kw['negative_prompt'] = negative
        try:
            return pipe(**kw).images[0]
        finally:
            # Libera la caché de PyTorch tras CADA generación (éxito o fallo): sin
            # esto la memoria "reservada" se va acumulando/fragmentando a lo largo
            # de una sesión larga con muchas generaciones seguidas, hasta que una
            # petición que debería caber revienta con CUDA OutOfMemoryError por
            # fragmentación, no por falta real de memoria (visto en pruebas reales).
            import gc
            gc.collect()
            torch.cuda.empty_cache()


chat_fn = None
if llm is not None:
    def chat_fn(messages, temperature, top_p, max_tokens, repeat_penalty):
        return llm.create_chat_completion(messages=messages, temperature=temperature,
                                          top_p=top_p, max_tokens=max_tokens,
                                          repeat_penalty=repeat_penalty)


model_name = 'z-image-turbo' if pipe is not None else 'qwen3-4b'
# refine_fn=None: Z-Image no soporta hires-fix (su refinar() local también es
# no-op, ver ZImageBackend.refinar) — RemoteBackend cae de vuelta a la imagen
# base sin refinar, sin romper nada.
app = build_app(model_name=model_name, generate_fn=generate_fn, chat_fn=chat_fn,
               refine_fn=None, api_key=API_KEY or None)
lanzar_con_tunel(app, puerto=8000)